# GeoLens + leafmap + samgeo

[GeoLens](https://github.com/geolens-io/geolens) is a self-hosted spatial data
hub: a catalog, search, and open standards over data that stays on your own
infrastructure. This notebook is a companion to
[`quickstart.ipynb`](quickstart.ipynb): that one tours every surface GeoLens
serves, this one goes deep on one of them and hands the result to
[samgeo](https://samgeo.gishub.org/) (`segment-geospatial`), which runs Meta's
Segment Anything Model over a georeferenced raster. leafmap and samgeo are
both maintained by [opengeos](https://github.com/opengeos) (Qiusheng Wu), so
segmenting a GeoLens-served raster on a leafmap map is the natural pairing.

The raster is real Sentinel-2 imagery that GeoLens imported *by reference*:
no file was uploaded into GeoLens's own storage, and its `raster-tiles` route
(TiTiler underneath) reads the pixels from the origin bucket at request time
rather than from a copy it made. That is a deliberate feature, not a
limitation — a generic STAC client can still open the original COG directly —
and section 2 below reads both hrefs off the same item to show it. The
demo's ["New York From Orbit"](https://demo.getgeolens.com/maps/1c4207ab-b1c0-4309-9924-c1ea355003a3)
map is built the same way, over the same import.

`GEOLENS` below defaults to the public demo, which needs no account and no
key. Pointing this at your own instance takes more than changing that one
line: the bbox and the assertions further down are calibrated to the demo's
own Sentinel-2 tile, so search `/api/stac/search` on your own instance and
substitute what it returns; expect to loosen or drop the assertions once
you're pointed elsewhere.


In [ ]:
# Pinned so a leafmap upgrade doesn't change what this notebook does out
# from under you. Safe to re-run: pip skips anything already at the pinned
# version. No geopandas here — everything this notebook reads is a STAC
# item (plain JSON) or raster tiles, never a feature collection.
%pip install -q leafmap==0.63.1 requests==2.33.1


In [ ]:
import os
import time

import leafmap.foliumap as leafmap
import requests

# Point this at your own instance to use your own catalog. The public demo
# answers every route below anonymously.
GEOLENS = "https://demo.getgeolens.com"
API = f"{GEOLENS}/api"

# Public datasets need no credentials, so the demo works with none set. For
# a private instance, export GEOLENS_API_KEY and every request below sends
# it as an X-Api-Key header.
API_KEY = os.environ.get("GEOLENS_API_KEY")
HEADERS = {"X-Api-Key": API_KEY} if API_KEY else {}

# The demo is one shared machine on the public internet, so a request
# occasionally times out or comes back 502 with nothing wrong at either end.
# quickstart.ipynb retries the same failures against this same demo;
# everything below goes through get_with_retry() instead of requests.get()
# for the same reason. Anything else in the 4xx range is a bad request, and
# retrying it just asks the same wrong question again.
RETRY_STATUS = {429, 500, 502, 503, 504}


def get_with_retry(url: str, attempts: int = 3, backoff: float = 1.0, **kwargs) -> requests.Response:
    kwargs.setdefault("timeout", 30)
    kwargs.setdefault("headers", HEADERS)
    for attempt in range(attempts):
        try:
            resp = requests.get(url, **kwargs)
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            reason = type(exc).__name__
        else:
            if resp.status_code not in RETRY_STATUS:
                resp.raise_for_status()
                return resp
            reason = f"HTTP {resp.status_code}"
        if attempt + 1 < attempts:
            print(f"  {reason}, retrying")
            time.sleep(backoff * 2**attempt)
    raise RuntimeError(f"{url} failed {attempts} times, last {reason}")


## 1. Find the imagery: search STAC by footprint

`GET /api/stac/search` is [STAC API - Item Search](https://docs.getgeolens.com/guides/api/ogc/#stac-10):
hand it a bbox (and, here, a collection) instead of a dataset id, and it
answers with whatever raster items overlap that footprint, the same query a
desktop STAC client or `pystac-client` would run. `geolens-unassigned` is
where GeoLens files a published raster that has not been assigned to a
collection yet — this Sentinel-2 tile is one of four sitting there. The bbox
below is a slice of Upper New York Bay narrow enough that only one of the
four intersects it.


In [ ]:
BBOX = [-74.05, 40.68, -73.95, 40.75]  # Upper New York Bay
COLLECTION = "geolens-unassigned"
# Pinned so a demo reset that swaps in a *different* single tile over this
# bbox fails loudly here instead of silently: section 2 and the gallery card,
# the sentinelNYHarbor fixture and the README all name this item and its
# Earth Search asset specifically, not "whichever one tile matches".
EXPECTED_ITEM_ID = "20683d4a-a531-4dab-ba80-c24e6d49ef76"  # Sentinel-2 TCI S2B_T18TWL_20260812T155330_L2A

resp = get_with_retry(f"{API}/stac/search", params={
    "collections": COLLECTION,
    "bbox": ",".join(str(n) for n in BBOX),
    "limit": 10,
})
items = resp.json()["features"]

print(f"{len(items)} STAC item(s) over {BBOX}:")
for feature in items:
    print(f"  {feature['id']}  {feature['properties']['title']!r}")

# One tile, not a scene mosaic: pick it by count rather than index so a
# regression to zero or several is a loud assertion, not a silent [0].
assert len(items) == 1, f"expected exactly one Sentinel-2 tile over this bbox, got {len(items)}"
item = items[0]
assert item["id"] == EXPECTED_ITEM_ID, (
    f"expected item {EXPECTED_ITEM_ID!r}, got {item['id']!r} ({item['properties']['title']!r}); "
    "the catalog moved, not just the count — the gallery card, the sentinelNYHarbor fixture "
    "and this notebook's README section all describe the item above specifically"
)


## 2. Two assets, one item

A normal GeoLens upload has exactly one raster asset: the `raster_tiles`
template GeoLens's own TiTiler pipeline serves. An item imported by
reference carries a second one, `data`, holding the origin catalog's own
asset href untouched — GeoLens neither re-hosts the pixels nor rewrites the
URL, since it is already public where it is. That is what lets a generic
STAC client (`stac-browser`, the QGIS STAC plugin, `rio-viz`, plain
`rasterio.open()`) read the same imagery GeoLens's own viewer draws.


In [ ]:
# The origin href is a stable S3 object key derived from the scene id, not a
# signed or expiring URL, so pinning it exactly catches the origin catalog
# re-publishing this scene under a different key just as loudly as it would
# catch GeoLens rehosting the pixels itself.
EXPECTED_ORIGIN_HREF = "https://e84-earth-search-sentinel-data.s3.us-west-2.amazonaws.com/sentinel-2-c1-l2a/18/T/WL/2026/8/S2B_T18TWL_20260812T155330_L2A/TCI.tif"

tiles_asset = item["assets"]["raster_tiles"]
origin_asset = item["assets"]["data"]

print("raster_tiles (GeoLens's TiTiler, reading the origin COG live):")
print(" ", tiles_asset["href"])
print("data (the origin catalog's own COG, untouched):")
print(" ", origin_asset["href"])

# The point of a by-reference import: the pixels were never copied onto
# GeoLens's own domain.
assert not origin_asset["href"].startswith(GEOLENS), "a by-reference item should still point at the origin catalog, not a GeoLens-hosted copy"
assert origin_asset["href"] == EXPECTED_ORIGIN_HREF, f"expected the origin COG at {EXPECTED_ORIGIN_HREF!r}, got {origin_asset['href']!r}"


## 3. Draw the GeoLens-served tiles on a leafmap map

Same `raster-tiles/{id}/tiles/{z}/{x}/{y}.png` template
[`quickstart.ipynb`](quickstart.ipynb) points at the Matterhorn DEM with —
the only difference is what is on the other end of it. TiTiler reads this
tile straight from the origin bucket named above on every request; nothing
about drawing it here needs to know that.


In [ ]:
tile_url = tiles_asset["href"]

# Confirm the route is live before wiring it into the map. This tile sits
# over the harbor mouth, inside the footprint on every zoom the map below
# opens at.
probe = get_with_retry(tile_url.format(z=12, x=1205, y=1540))
assert probe.headers["content-type"] == "image/png"

m = leafmap.Map(center=[40.70, -74.01], zoom=11)
m.add_tile_layer(
    url=tile_url,
    name=item["properties"]["title"],
    attribution="Contains modified Copernicus Sentinel data, ESA/Copernicus via Element 84 Earth Search, imported by reference into the GeoLens demo",
)
m


## 4. Optional: segment the harbor with samgeo

[`segment-geospatial`](https://samgeo.gishub.org/) runs Meta's Segment
Anything Model over a georeferenced raster. It needs `torch` and a
multi-hundred-megabyte model checkpoint, neither of which belongs in a
notebook that is supposed to run anywhere in a few seconds, so this cell is
off by default — the same convention `quickstart.ipynb` uses for its own
optional samgeo section. Flip the flag and install the extra once, and it
pulls the tiles above into a local GeoTIFF and segments it. Real true-colour
imagery is what SAM is built for, more so than the colour-relief DEM
`quickstart.ipynb` segments: expect cleaner boundaries between water, piers
and the vessels visible on the water in this scene.


In [ ]:
RUN_SEGMENTATION = False  # pip install segment-geospatial torch, then flip this on

if RUN_SEGMENTATION:
    from samgeo import SamGeo

    # samgeo segments a raster file, not a live tile source, so pull the
    # tiles over one harbor tile's footprint into a local GeoTIFF first, the
    # same way quickstart.ipynb does for the Matterhorn DEM.
    harbor_bbox = [-74.09, 40.65, -74.00, 40.71]
    leafmap.tms_to_geotiff("harbor.tif", harbor_bbox, zoom=14, source=tile_url, to_cog=True)

    sam = SamGeo(model_type="vit_h", automatic=True)
    sam.generate("harbor.tif", output="harbor_segments.tif")
    m.add_raster("harbor_segments.tif", layer_name="Segments", opacity=0.6)
    m
else:
    print("Skipping: RUN_SEGMENTATION is False. See the README for what this needs.")


---

**Data.** Contains modified Copernicus Sentinel data, processed by
ESA/Copernicus and served through [Element 84's Earth
Search](https://element84.com/earth-search/) STAC API; GeoLens imported this
tile by reference rather than re-hosting it.

Read further: [`quickstart.ipynb`](quickstart.ipynb) tours the rest of what
GeoLens serves, the [STAC API reference](https://docs.getgeolens.com/guides/api/ogc/#stac-10)
covers the routes above, and ["New York From Orbit"](https://demo.getgeolens.com/maps/1c4207ab-b1c0-4309-9924-c1ea355003a3)
is a saved map built from the same by-reference import. If GeoLens is useful
to you, [star it on GitHub](https://github.com/geolens-io/geolens). That's
how most people find it.
